In [1]:
from src import pre_processor_inferencia
import joblib
import pandas as pd
from scipy.stats import norm
import random

In [2]:
def pipeline_aluno(dt_presenca, dt_desempenho, dados_aluno):
    
    df = pd.DataFrame([dados_aluno])
    X  = pre_processor_inferencia(df)

    # Probabilidades de presença
    prob_presenca = dt_presenca.predict_proba(X)[0]
    presenca      = dt_presenca.predict(X)[0]

    resultado = {
        'presenca':         'presente' if presenca == 1 else 'ausente',
        'prob_presente':    f'{prob_presenca[1]:.1%}',
        'prob_ausente':     f'{prob_presenca[0]:.1%}',
        'desempenho':       None,
        'prob_acima':       None,
        'prob_abaixo':      None,
    }

    if presenca == 1:
        prob_desempenho = dt_desempenho.predict_proba(X)[0]
        desempenho      = dt_desempenho.predict(X)[0]

        resultado['desempenho']  = 'acima da mediana' if desempenho == 1 else 'abaixo da mediana'
        resultado['prob_acima']  = f'{prob_desempenho[1]:.1%}'
        resultado['prob_abaixo'] = f'{prob_desempenho[0]:.1%}'

    return resultado

In [15]:
def chances_por_curso(prob_acima: float, desvio=0.10):
    cursos = {
    'Medicina (top)'        : 0.97, 
    'Direito (top)'         : 0.90,
    'Computação (top)'      : 0.85,
    'Engenharia'            : 0.74,
    'Administração'         : 0.65,
    'Pedagogia'             : 0.54,
    'Licenciaturas'         : 0.53,
    'Tecnólogos'            : 0.49,
    'Cursos noturnos'       : 0.42,
}
    resultado = {}
    for curso, percentil_corte in cursos.items():
        # Probabilidade do aluno superar o corte dado a incerteza
        z = (prob_acima - percentil_corte) / desvio
        chance = norm.cdf(z)
        resultado[curso] = f'{chance:.1%}'
        
    return resultado

In [16]:
def gerar_aluno_aleatorio():
    return {
        'Q001': random.choice(list('ABCDEFGH')),
        'Q002': random.choice(list('ABCDEFGH')),
        'Q003': random.choice(list('ABCDEF')),
        'Q004': random.choice(list('ABCDEF')),
        'Q005': random.randint(1,20 ),
        'Q006': random.choice(list('ABCDEFGHIJKLMNOPQ')),
        'Q007': random.choice(list('ABCD')),
        'Q008': random.choice(list('ABCDE')),
        'Q009': random.choice(list('ABCDE')),
        'Q010': random.choice(list('ABCDE')),
        'Q011': random.choice(list('ABCDE')),
        'Q012': random.choice(list('ABCDE')),
        'Q013': random.choice(list('ABCDE')),
        'Q014': random.choice(list('ABCDE')),
        'Q015': random.choice(list('ABCDE')),
        'Q016': random.choice(list('ABCDE')),
        'Q017': random.choice(list('ABCDE')),
        'Q018': random.choice(list('ABCDE')),
        'Q019': random.choice(list('ABCDE')),
        'Q020': random.choice(list('ABCDE')),
        'Q021': random.choice(list('ABCDE')),
        'Q022': random.choice(list('ABCDE')),
        'Q023': random.choice(list('ABCDE')),
        'Q024': random.choice(list('ABCDE')),
        'Q025': random.choice(list('AB')),
        'TP_FAIXA_ETARIA':        random.randint(1, 10),
        'TP_ESTADO_CIVIL':        random.randint(1, 4),
        'TP_ESCOLA':              random.choice([2, 3]),
        'TP_ST_CONCLUSAO':        random.randint(1, 4),
        'IN_TREINEIRO':           random.choice([0, 1]),
        'NU_ANO':                 random.choice([2019, 2020, 2021, 2022, 2023]),
        'TP_DEPENDENCIA_ADM_ESC': random.randint(1, 4),
        'TP_LOCALIZACAO_ESC':     random.choice([1, 2]),
        'TP_SIT_FUNC_ESC':        random.randint(1, 3),
    }

In [17]:
dt_presenca   = joblib.load('models/dt_presenca.pkl')
rf_desempenho = joblib.load('models/rf_desempenho.pkl')

c:\Users\sergi\OneDrive\Documentos\GitHub\Projeto_ENEM\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\sergi\OneDrive\Documentos\GitHub\Projeto_ENEM\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [18]:
resultados = []

for i in range(10):
    aluno = gerar_aluno_aleatorio()
    resultado = pipeline_aluno(dt_presenca, rf_desempenho, aluno)
    resultados.append(resultado)
    print(f'Aluno {i+1}: {resultado}')

Aluno 1: {'presenca': 'ausente', 'prob_presente': '13.3%', 'prob_ausente': '86.7%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 2: {'presenca': 'presente', 'prob_presente': '86.7%', 'prob_ausente': '13.3%', 'desempenho': 'abaixo da mediana', 'prob_acima': '45.4%', 'prob_abaixo': '54.6%'}
Aluno 3: {'presenca': 'presente', 'prob_presente': '64.6%', 'prob_ausente': '35.4%', 'desempenho': 'abaixo da mediana', 'prob_acima': '16.2%', 'prob_abaixo': '83.8%'}
Aluno 4: {'presenca': 'presente', 'prob_presente': '86.7%', 'prob_ausente': '13.3%', 'desempenho': 'abaixo da mediana', 'prob_acima': '42.5%', 'prob_abaixo': '57.5%'}
Aluno 5: {'presenca': 'ausente', 'prob_presente': '31.9%', 'prob_ausente': '68.1%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 6: {'presenca': 'ausente', 'prob_presente': '31.9%', 'prob_ausente': '68.1%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 7: {'presenca': 'ausente', 'prob_presente': '13.3%', 'prob

In [19]:
alunos_acima = [r for r in resultados if r['desempenho'] != None]
print(f'\nAlunos com desempenho acima da mediana: {len(alunos_acima)}')


Alunos com desempenho acima da mediana: 4


In [20]:
alunos_acima

[{'presenca': 'presente',
  'prob_presente': '86.7%',
  'prob_ausente': '13.3%',
  'desempenho': 'abaixo da mediana',
  'prob_acima': '45.4%',
  'prob_abaixo': '54.6%'},
 {'presenca': 'presente',
  'prob_presente': '64.6%',
  'prob_ausente': '35.4%',
  'desempenho': 'abaixo da mediana',
  'prob_acima': '16.2%',
  'prob_abaixo': '83.8%'},
 {'presenca': 'presente',
  'prob_presente': '86.7%',
  'prob_ausente': '13.3%',
  'desempenho': 'abaixo da mediana',
  'prob_acima': '42.5%',
  'prob_abaixo': '57.5%'},
 {'presenca': 'presente',
  'prob_presente': '86.7%',
  'prob_ausente': '13.3%',
  'desempenho': 'acima da mediana',
  'prob_acima': '62.0%',
  'prob_abaixo': '38.0%'}]

In [21]:
for i, aluno in enumerate(alunos_acima):
    prob = float(aluno['prob_acima'].strip('%')) / 100
    
    print(f'\n===== Aluno {i+1} =====')
    print(f"Presença: {aluno['presenca']} ({aluno['prob_presente']})")
    print(f"Desempenho: {aluno['desempenho']} | Prob. acima: {aluno['prob_acima']}")
    print(f"\nChances por curso:")
    
    for curso, chance in chances_por_curso(prob).items():
        print(f'  {curso:<25} {chance}')



===== Aluno 1 =====
Presença: presente (86.7%)
Desempenho: abaixo da mediana | Prob. acima: 45.4%

Chances por curso:
  Medicina (top)            0.0%
  Direito (top)             0.0%
  Computação (top)          0.0%
  Engenharia                0.2%
  Administração             2.5%
  Pedagogia                 19.5%
  Licenciaturas             22.4%
  Tecnólogos                35.9%
  Cursos noturnos           63.3%

===== Aluno 2 =====
Presença: presente (64.6%)
Desempenho: abaixo da mediana | Prob. acima: 16.2%

Chances por curso:
  Medicina (top)            0.0%
  Direito (top)             0.0%
  Computação (top)          0.0%
  Engenharia                0.0%
  Administração             0.0%
  Pedagogia                 0.0%
  Licenciaturas             0.0%
  Tecnólogos                0.1%
  Cursos noturnos           0.5%

===== Aluno 3 =====
Presença: presente (86.7%)
Desempenho: abaixo da mediana | Prob. acima: 42.5%

Chances por curso:
  Medicina (top)            0.0%
  Direito (t